In [ ]:
from string import Template
from pmtools.runner import Engine
from pmtools.kernels import calculate_stacking_fraction
from pmtools.refractored_toolbox import assemble_paths, determine_box_dim_from_filename

master_dict={}
master_dict['what_am_I_looking_at'] = ('chains/ligands_bending',)
master_dict['which_sim'] = ('sim_1','sim_2','sim_3','sim_4','sim_5','sim_6','sim_7', 'sim_8')
master_dict['which_quartet_number'] = ('6120',)
master_dict['which_number_per_quartet'] = ('25',)
master_dict['which_crowder'] = ('4080', )
master_dict['which_concentration'] = ('0.6',)
master_dict['which_vdW'] = ('5.0',)
master_dict['which_vdW_ligand'] = ('5.0','10.0','15.0','20.0')
master_dict['what_monomer_number'] = ('15',)
master_dict['what_monomer_number_ligand'] = ('1','3','5')

template_hndl=Template(
    "$what_am_I_looking_at/$which_sim/custom_data_wip_${which_quartet_number}_${which_number_per_quartet}_${which_crowder}_${which_concentration}_${which_vdW}_${which_vdW_ligand}_${what_monomer_number}_${what_monomer_number_ligand}.h5"
    )

keys_assembly,paths_accordingly=assemble_paths(master_dict,template_hndl,'which_sim')
kernel_kwargs={'chunk': (-5, None, 1), 'box_dim': determine_box_dim_from_filename(template_hndl, paths_accordingly[0][0], key_concentration='which_concentration', key_obj_no='which_quartet_number'), 'crit': 1.3, 'extra_flag': 'concentration_variation'}
engine_inst=Engine(world_path='$FULLPATH') # $FULLPATH should point to the directory where the data is so that appending what_am_I_looking_at and which_sim will lead to the correct path
engine_inst.assemble_paths(master_dict,template_hndl,'which_sim')
engine_inst.register_kernel(calculate_stacking_fraction, **kernel_kwargs)
engine_inst.run(max_workers=16)
engine_inst.collect_results(kill_workers=False)
engine_inst.save_results(filename='engine_apptainer_segments_ligands_bend')
engine_inst.shutdown()
print('DONE!')